In [1]:
import os
import opf

import numpy as np
import pandas as pd
import pyomo.environ as pyo

from matpower import path_matpower_cases, start_instance
from matpowercaseframes import CaseFrames


In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 300)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
case_path = os.path.join(path_matpower_cases, 'case9.m')

In [4]:
model = opf.build_model('acopf')
network = opf.parse_file(case_path)
model.instantiate(network)
result = model.solve(
    solver_option={'print_level' : 5},
    tee=True
)

build model... end
instantiate model... end
Ipopt 3.14.16: print_level=5


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.16, running with linear solver MUMPS 5.6.2.

Number of nonzeros in equality constraint Jacobian...:      223
Number of nonzeros in inequality constraint Jacobian.:       54
Number of nonzeros in Lagrangian Hessian.............:      102

Total number of variables............................:       60
                     variables with only lower bounds:        0
                variables with lower and upper bounds:       15
                     variables with only upper bounds:        0
Total number of eq

In [5]:
print(
    f"Status: {result['termination_status']}\n"
    f"Objective function: {result['obj_cost']}\n"
    f"Output power: {result['sol']['primal']['pg']}\n"
    f"Computation time: {result['time']}"
)

Status: optimal
Objective function: 5296.686202481215
Output power: {'1': 0.8979870766526404, '2': 1.3432060074581076, '3': 0.9418738042409803}
Computation time: 0.033503055572509766


In [6]:
cf = CaseFrames(case_path)
cf.infer_numpy()

In [7]:
def short_int_df_index(df):
    df.index = [int(i) for i in df.index]
    return df.sort_index()

df_pyopf_branch = short_int_df_index(pd.DataFrame({
    'pf_from': result['sol']['primal']['pf_from'], 
    'pf_to': result['sol']['primal']['pf_to'],
}))
df_pyopf_gen = short_int_df_index(pd.DataFrame({
    'pg': result['sol']['primal']['pg'],
    'qg': result['sol']['primal']['qg'],
}))
df_pyopf_bus = short_int_df_index(pd.DataFrame({
    'vm': result['sol']['primal']['vm'],
    'va': result['sol']['primal']['va'],
}))
df_pyopf_branch[['pf_from', 'pf_to']] = df_pyopf_branch[['pf_from', 'pf_to']] * cf.baseMVA
df_pyopf_gen[['pg', 'qg']] = df_pyopf_gen[['pg', 'qg']] * cf.baseMVA
df_pyopf_bus['va'] = df_pyopf_bus['va'] * 180 / np.pi

# print(df_pyopf_branch)
# print(df_pyopf_gen)
# print(df_pyopf_bus)

cf.branch[['PF', 'PT']] = df_pyopf_branch[['pf_from', 'pf_to']]
cf.bus[['VM', 'VA']] = df_pyopf_bus[['vm', 'va']]
cf.gen[['PG', 'QG']] = df_pyopf_gen[['pg', 'qg']]

In [8]:
print(cf.branch)
print(cf.bus)
print(cf.gen)

   F_BUS  T_BUS  BR_R  BR_X  BR_B  RATE_A  RATE_B  RATE_C  TAP  SHIFT  BR_STATUS  ANGMIN  ANGMAX    PF    PT
1      1      4  0.00  0.06  0.00     250     250     250    0      0          1    -360     360  0.90 -0.90
2      4      5  0.02  0.09  0.16     250     250     250    0      0          1    -360     360  0.35 -0.35
3      5      6  0.04  0.17  0.36     150     150     150    0      0          1    -360     360 -0.55  0.56
4      3      6  0.00  0.06  0.00     300     300     300    0      0          1    -360     360  0.94 -0.94
5      6      7  0.01  0.10  0.21     150     150     150    0      0          1    -360     360  0.38 -0.38
6      7      8  0.01  0.07  0.15     250     250     250    0      0          1    -360     360 -0.62  0.62
7      8      2  0.00  0.06  0.00     250     250     250    0      0          1    -360     360 -1.34  1.34
8      8      9  0.03  0.16  0.31     250     250     250    0      0          1    -360     360  0.72 -0.71
9      9      4  0.

In [9]:
m = start_instance()

In [10]:
sol = m.runpf(cf.to_dict(), verbose=False)


In [11]:
m.exit()

In [12]:
cf_runpf = CaseFrames(sol)
print(cf_runpf.branch)
print(cf_runpf.bus)
print(cf_runpf.gen)

   F_BUS  T_BUS  BR_R  BR_X  BR_B  RATE_A  RATE_B  RATE_C  TAP  SHIFT  BR_STATUS  ANGMIN  ANGMAX      PF     QF     PT     QT
1   1.00   4.00  0.00  0.06  0.00  250.00  250.00  250.00 0.00   0.00       1.00 -360.00  360.00   90.31  25.08 -90.31 -20.40
2   4.00   5.00  0.02  0.09  0.16  250.00  250.00  250.00 0.00   0.00       1.00 -360.00  360.00   35.30   0.23 -35.09 -15.54
3   5.00   6.00  0.04  0.17  0.36  150.00  150.00  150.00 0.00   0.00       1.00 -360.00  360.00  -54.91 -14.46  56.06 -18.05
4   3.00   6.00  0.00  0.06  0.00  300.00  300.00  300.00 0.00   0.00       1.00 -360.00  360.00   94.19 -11.52 -94.19  16.54
5   6.00   7.00  0.01  0.10  0.21  150.00  150.00  150.00 0.00   0.00       1.00 -360.00  360.00   38.13   1.51 -37.95 -21.94
6   7.00   8.00  0.01  0.07  0.15  250.00  250.00  250.00 0.00   0.00       1.00 -360.00  360.00  -62.05 -13.06  62.37   0.21
7   8.00   2.00  0.00  0.06  0.00  250.00  250.00  250.00 0.00   0.00       1.00 -360.00  360.00 -134.32   8.29 134.32